# Day 062 — Exercise 1: assert_response

Repeating the same assertion boilerplate in every API test gets tedious and produces unhelpful failure messages. A reusable helper encapsulates the pattern and produces informative errors when tests fail.

Without a helper:
```python
assert r.status_code == 200  # ❌ only shows 'False', not actual code
```

With `assert_response`:
```python
data = assert_response(r, 200, ('status', 'version'))
# ❌ Expected 200, got 500: {"detail": "Internal Server Error"}
```

In [ ]:
# --- minimal app for testing ---
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

def _build_echo_app():
    app = FastAPI()
    class _Body(BaseModel):
        text: str = Field(min_length=1)
    @app.get("/health")
    def health():
        return {"status": "ok"}
    @app.post("/echo")
    def echo(body: _Body):
        return {"echo": body.text}
    return app

_client = TestClient(_build_echo_app(), raise_server_exceptions=False)


In [ ]:
# no extra imports needed


## Task

Implement `assert_response(response, expected_status, required_keys=()) -> dict`:

1. `assert response.status_code == expected_status` — include actual status + `response.text[:200]` in the failure message
2. `data = response.json()` — parse the body
3. For each key in `required_keys`: assert it's in `data`, name the missing key
4. Return `data`

## Your Implementation

In [ ]:
def assert_response(response, expected_status: int, required_keys: tuple = ()) -> dict:
    """Assert response status and that all required_keys appear in JSON body.

    Returns the parsed JSON dict on success.
    Raises AssertionError with a clear message on:
    - wrong status code (include response.text[:200] in the message)
    - any required key missing from the response body

    Hint: f"{expected_status}, got {response.status_code}: {response.text[:200]}"
    """
    # TODO: assert status, parse JSON, assert each required key, return dict
    raise NotImplementedError


In [ ]:
def assert_response(response, expected_status: int, required_keys: tuple = ()) -> dict:
    assert response.status_code == expected_status, (
        f"Expected {expected_status}, got {response.status_code}: {response.text[:200]}")
    data = response.json()
    for key in required_keys:
        assert key in data, f"Missing key {key!r} in response: {data}"
    return data


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # 1. returns dict on correct status
    r = _client.get("/health")
    data = assert_response(r, 200, ("status",))
    assert isinstance(data, dict) and data.get("status") == "ok"
    score += 1; print("\u2705 returns parsed JSON dict on correct status + key")

    # 2. correct status, no required_keys
    r2 = _client.post("/echo", json={"text": "hello"})
    d2 = assert_response(r2, 200)
    assert d2["echo"] == "hello"
    score += 1; print("\u2705 works with no required_keys")

    # 3. wrong status → AssertionError with message containing the actual code
    r3 = _client.get("/health")
    try:
        assert_response(r3, 404)
        assert False, "Should have raised AssertionError"
    except AssertionError as e:
        assert "200" in str(e) or "404" in str(e), f"Error message missing status codes: {e}"
    score += 1; print("\u2705 wrong status raises AssertionError with status info")

    # 4. missing key → AssertionError
    r4 = _client.get("/health")
    try:
        assert_response(r4, 200, ("missing_key",))
        assert False, "Should have raised AssertionError"
    except AssertionError as e:
        assert "missing_key" in str(e), f"Error message should name missing key: {e}"
    score += 1; print("\u2705 missing required key raises AssertionError naming the key")

    # 5. 422 assertion (empty text)
    r5 = _client.post("/echo", json={"text": ""})
    assert_response(r5, 422)
    score += 1; print("\u2705 correctly asserts 422 validation error")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def assert_response(response, expected_status: int, required_keys: tuple = ()) -> dict:
    assert response.status_code == expected_status, (
        f"Expected {expected_status}, got {response.status_code}: {response.text[:200]}")
    data = response.json()
    for key in required_keys:
        assert key in data, f"Missing key {key!r} in response: {data}"
    return data
```

**Why `response.text[:200]`?** A bare `assert r.status_code == 200` prints nothing useful when it fails. Including the first 200 chars of the response body shows you the actual error — e.g. `{"detail": "Internal Server Error"}` — so you know where to look without re-running with extra print statements.

</details>